[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IyadSultan/CCI/blob/main/session6/student/Lab1_Basic_RAG_LlamaParse_Student.ipynb)


# Lab 1: Basic RAG with LlamaParse
## CCI Session 6

**Duration:** 15 minutes

### Clinical Scenario
> KHCC pediatric oncologists need quick answers from the National Wilms Tumor treatment guidelines (200+ pages with figures, tables, complex layouts). You'll build a RAG system that parses the PDF, embeds it, and answers clinical questions.

### Objective
- Parse a complex clinical PDF with LlamaParse
- Chunk and embed the content
- Store in ChromaDB
- Build a retrieval QA chain
- Test with real clinical questions

---
## Setup — install packages

After installs, **Runtime → Restart session** if Colab still raises import errors.


In [ ]:
!pip install -q llama-parse llama-index langchain langchain-openai langchain-community langchain-chroma chromadb pypdf

## API Keys (Colab `userdata`)

In [ ]:
import os
from google.colab import userdata

os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
os.environ['LLAMA_CLOUD_API_KEY'] = userdata.get('LLAMA_CLOUD_API_KEY')
print('Keys loaded.')

## Upload `WT.pdf` to Colab
Run the cell, click **Choose Files** and pick the Wilms tumor PDF.

In [ ]:
from google.colab import files
uploaded = files.upload()  # upload WT.pdf
PDF_PATH = '/content/WT.pdf'

---
## Cell 1 — Parse with LlamaParse

Configure **`LlamaParse`** with `result_type='markdown'` so the guideline text keeps structure (tables, headings). Compare visually with PyPDF in the next section.


In [ ]:
from llama_parse import LlamaParse

# TODO: create a LlamaParse parser with result_type='markdown'
# Hint: parser = LlamaParse(api_key=os.environ['LLAMA_CLOUD_API_KEY'], result_type='markdown')
parser = ...

# TODO: parse the PDF
# Hint: documents = parser.load_data(PDF_PATH)
documents = ...

print(f'Number of documents: {len(documents)}')
print('--- Preview ---')
print(documents[0].text[:1500])

## Cell 2 — Compare with naive PyPDF
Notice how tables / figure captions get mangled by the naive loader.

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

# TODO: load with PyPDFLoader
# Hint: naive_docs = PyPDFLoader(PDF_PATH).load()
naive_docs = ...

print(f'Pages (PyPDF): {len(naive_docs)}')
print('--- PyPDF sample (note jumbled tables) ---')
print(naive_docs[5].page_content[:1500])

print('\n--- LlamaParse sample (clean markdown tables) ---')
print(documents[0].text[1500:3000])

---
## Cell 3 — Chunking for RAG

Split the parsed markdown into **chunks** before embedding. Use **`RecursiveCharacterTextSplitter`** so splits prefer paragraph and sentence boundaries.

**Design choices:**
- **`chunk_size`**: target characters per chunk (try 800–1200 for guidelines).
- **`chunk_overlap`**: copied characters between adjacent chunks so facts on boundaries are not lost.
- Alternatives include fixed windows or section-based splitting; this lab uses the recursive splitter.


In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

# Convert LlamaParse docs to LangChain Documents
lc_docs = [Document(page_content=d.text, metadata={'source': 'WT.pdf'}) for d in documents]

# TODO: build a RecursiveCharacterTextSplitter with chunk_size=1000, chunk_overlap=200
splitter = ...

# TODO: split the documents
chunks = ...

print(f'Number of chunks: {len(chunks)}')
print('--- Sample chunk ---')
print(chunks[10].page_content[:600])

---
## Cell 4 — Embedding model

We use **`text-embedding-3-small`** for a good balance of quality and cost. Same model must be used when **building** the index and when **querying** it.

**You could compare:** `text-embedding-3-large` (stronger, pricier) or other providers — always re-embed the whole corpus if you change model.


In [ ]:
from langchain_openai import OpenAIEmbeddings

# TODO: embeddings = OpenAIEmbeddings(model='text-embedding-3-small')
embeddings = ...

---
## Cell 5 — Vector store (Chroma)

**Chroma** stores vectors on disk under `persist_directory` so you can restart the runtime without re-embedding.

**Other stacks:** FAISS (fast, local), or hosted vector DBs for production multi-user search.

The **retriever** wraps similarity search; `k` is how many chunks the LLM sees as context.


In [ ]:
from langchain_chroma import Chroma

# TODO: create vectorstore from chunks + embeddings
# Hint: Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory='./chroma_wt')
vectorstore = ...

retriever = vectorstore.as_retriever(search_kwargs={'k': 4})

## Cell 6 — Build RAG Chain

In [ ]:
from langchain_openai import ChatOpenAI
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)

prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are a clinical assistant. Answer the question using ONLY the provided context. If unknown, say so.\n\nContext:\n{context}'),
    ('human', '{input}')
])

# TODO: build the doc chain and retrieval chain
# Hint:
# doc_chain = create_stuff_documents_chain(llm, prompt)
# rag_chain = create_retrieval_chain(retriever, doc_chain)
doc_chain = ...
rag_chain = ...

q1 = 'What is the standard treatment for Stage III favorable histology Wilms tumor?'
q2 = 'What are the most common side effects of vincristine?'

for q in [q1, q2]:
    print(f'\nQ: {q}')
    res = rag_chain.invoke({'input': q})
    print('A:', res['answer'])

## Cell 7 — Inspect Retrieved Chunks (Citations)

In [ ]:
res = rag_chain.invoke({'input': q1})
for i, doc in enumerate(res['context']):
    print(f'--- Chunk {i+1} ---')
    print(doc.page_content[:300])
    print()

## Stretch Challenge
Try `chunk_size` of 500 and 2000. Re-run question 1. Which produces better citations?

## KHCC Connection
This pattern is the foundation for any internal KHCC document QA system: clinical pathways, drug formulary, IRB SOPs. LlamaParse keeps tables and figures intact — critical for guideline lookup.